Etapa	Qué vas a aprender
1. Conexión	Conectar Python con tu base de datos.
2. Extracción	Ejecutar consultas SQL desde un script de Python.
3. Análisis	Trabajar con los resultados usando Pandas.
4. Automatización	Ejecutar las consultas y los cálculos sin intervención manual.
5. Alertas	Enviar un correo cuando se cumpla una condición.
6. Programación	Configurar la ejecución periódica del proceso.

In [5]:
import pandas as pd
import sqlalchemy
import psycopg
from dotenv import load_dotenv
import os
from pathlib import Path

import pandas as pd

from sqlalchemy import create_engine, text

print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("Conexión PostgreSQL: librerías disponibles")

Pandas: 3.0.1
SQLAlchemy: 2.0.48
Conexión PostgreSQL: librerías disponibles


In [24]:
import sys
print(sys.executable)

d:\Josefina\Proyectos\Datascience\.venv\Scripts\python.exe


In [ ]:
# %pip install "psycopg[binary]"

In [6]:


# Cargar las variables del archivo .env
load_dotenv()

# Leer los datos de conexión
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

# Comprobar que estén configurados sin mostrar la contraseña
assert all([host, port, database, user, password]), (
    "Faltan variables de conexión en el archivo .env"
)

# Crear la conexión
connection_url = (
    f"postgresql+psycopg://{user}:{password}"
    f"@{host}:{port}/{database}"
)

engine = create_engine(connection_url)

# Probar la conexión
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user, version()")
    )
    database_name, username, pg_version = result.fetchone()

print("Conexión exitosa")
print("Base de datos:", database_name)
print("Usuario:", username)
print("Servidor:", pg_version.split(",")[0])

Conexión exitosa
Base de datos: olist
Usuario: postgres
Servidor: PostgreSQL 18.4 on x86_64-windows


In [7]:
query = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
  AND table_schema NOT IN ('pg_catalog', 'information_schema')
ORDER BY table_schema, table_name;
"""

tablas = pd.read_sql(query, engine)

display(tablas)

,table_schema,table_name
0,public,customers
1,public,geolocation
2,public,order_items
3,public,order_payments
4,public,order_reviews
5,public,orders
6,public,product_category_name_translation
7,public,products
8,public,sellers


In [8]:
query = """
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN (
      'orders',
      'order_items',
      'order_payments',
      'customers'
  )
ORDER BY table_name, ordinal_position;
"""

columnas = pd.read_sql(query, engine)

display(columnas)

,table_name,column_name,data_type
0,customers,customer_id,character varying
1,customers,customer_unique_id,character varying
2,customers,customer_zip_code_prefix,integer
3,customers,customer_city,character varying
4,customers,customer_state,character varying
5,order_items,order_id,character varying
6,order_items,order_item_id,integer
7,order_items,product_id,character varying
8,order_items,seller_id,character varying
9,order_items,shipping_limit_date,timestamp without time zone


In [9]:
# conocer los estados de los pedidos
query = """
SELECT
    order_status,
    COUNT(*) AS cantidad_pedidos
FROM orders
GROUP BY order_status
ORDER BY cantidad_pedidos DESC;
"""

estados_pedidos = pd.read_sql(query, engine)

display(estados_pedidos)

,order_status,cantidad_pedidos
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [10]:
# consulta SQL que calcule los indicadores por mes. Así empezamos a construir la lógica que después automatizaremos.
query = """
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp)::date AS mes,
    COUNT(DISTINCT o.order_id) AS pedidos_entregados,
    SUM(oi.price) AS facturacion_productos,
    SUM(oi.price) / NULLIF(COUNT(DISTINCT o.order_id), 0)
        AS ticket_promedio
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY DATE_TRUNC('month', o.order_purchase_timestamp)::date
ORDER BY mes;
"""

ventas_mensuales = pd.read_sql(query, engine)

display(ventas_mensuales.head(10))

,mes,pedidos_entregados,facturacion_productos,ticket_promedio
0,2016-09-01,1,134.97,134.970000
1,2016-10-01,265,40325.11,152.170226
2,2016-12-01,1,10.90,10.900000
3,2017-01-01,750,111798.36,149.064480
4,2017-02-01,1653,234223.40,141.695947
5,2017-03-01,2546,359198.85,141.083602
6,2017-04-01,2303,340669.68,147.924307
7,2017-05-01,3546,489338.25,137.997250
8,2017-06-01,3135,421923.37,134.584807
9,2017-07-01,3872,481604.52,124.381333


In [11]:
# Mostrar los últimos 12 meses disponibles
display(ventas_mensuales.tail(12))

,mes,pedidos_entregados,facturacion_productos,ticket_promedio
11,2017-09-01,4150,607399.67,146.361366
12,2017-10-01,4478,648247.65,144.762762
13,2017-11-01,7289,987765.37,135.514525
14,2017-12-01,5513,726033.19,131.694756
15,2018-01-01,7069,924645.00,130.802801
16,2018-02-01,6555,826437.13,126.077365
17,2018-03-01,7003,953356.25,136.135406
18,2018-04-01,6798,973534.09,143.208898
19,2018-05-01,6749,977544.69,144.842894
20,2018-06-01,6099,856077.86,140.363643


In [ ]:
# conocer el rango de fechas de los pedidos:
query_fechas = """
SELECT
    MIN(order_purchase_timestamp) AS primera_compra,
    MAX(order_purchase_timestamp) AS ultima_compra,
    COUNT(*) AS total_pedidos
FROM orders;
"""

fechas_olist = pd.read_sql(query_fechas, engine)

display(fechas_olist)

,primera_compra,ultima_compra,total_pedidos
0,2016-09-04 21:15:19,2018-10-17 17:30:18,99441


In [13]:
# comprobar qué pasó en septiembre y octubre
query_estados_mensuales = """
SELECT
    DATE_TRUNC('month', order_purchase_timestamp)::date AS mes,
    order_status,
    COUNT(*) AS cantidad_pedidos
FROM orders
WHERE order_purchase_timestamp >= '2018-08-01'
GROUP BY
    DATE_TRUNC('month', order_purchase_timestamp)::date,
    order_status
ORDER BY mes, order_status;
"""

estados_mensuales = pd.read_sql(query_estados_mensuales, engine)

display(estados_mensuales)

,mes,order_status,cantidad_pedidos
0,2018-08-01,canceled,84
1,2018-08-01,delivered,6351
2,2018-08-01,invoiced,23
3,2018-08-01,shipped,47
4,2018-08-01,unavailable,7
5,2018-09-01,canceled,15
6,2018-09-01,shipped,1
7,2018-10-01,canceled,4


In [15]:
# Construir primera alerta automática

# Copiar y ordenar los datos mensuales
monitoreo = ventas_mensuales.copy()
monitoreo["mes"] = pd.to_datetime(monitoreo["mes"])
monitoreo = monitoreo.sort_values("mes")

# Conservar meses con suficientes pedidos entregados
monitoreo = monitoreo[
    (monitoreo["mes"] >= "2017-01-01")
    & (monitoreo["mes"] <= "2018-08-01")
    & (monitoreo["pedidos_entregados"] >= 1000)
].copy()

monitoreo = monitoreo.sort_values("mes").copy()

# Crear una tabla de facturación indexada por mes
facturacion_por_mes = monitoreo.set_index("mes")[
    "facturacion_productos"
]

# Buscar la facturación del mes calendario anterior
mes_anterior = monitoreo["mes"] - pd.DateOffset(months=1)

monitoreo["facturacion_mes_anterior"] = mes_anterior.map(
    facturacion_por_mes
)

# Calcular la variación solo cuando existe el mes anterior
monitoreo["variacion_pct"] = (
    (
        monitoreo["facturacion_productos"]
        / monitoreo["facturacion_mes_anterior"]
    ) - 1
) * 100

# Activar alertas únicamente si hay una comparación válida
umbral_caida = -15

monitoreo["alerta"] = (
    monitoreo["variacion_pct"].notna()
    & (monitoreo["variacion_pct"] < umbral_caida)
)

display(
    monitoreo[
        [
            "mes",
            "pedidos_entregados",
            "facturacion_productos",
            "facturacion_mes_anterior",
            "variacion_pct",
            "alerta",
        ]
    ].round(2)
)

C:\Users\Josefina Urquiza\AppData\Local\Temp\ipykernel_14472\1989061939.py:55: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,mes,pedidos_entregados,facturacion_productos,facturacion_mes_anterior,variacion_pct,alerta
4,2017-02-01,1653,234223.40,NaN,NaN,False
5,2017-03-01,2546,359198.85,234223.40,53.36,False
6,2017-04-01,2303,340669.68,359198.85,-5.16,False
7,2017-05-01,3546,489338.25,340669.68,43.64,False
8,2017-06-01,3135,421923.37,489338.25,-13.78,False
9,2017-07-01,3872,481604.52,421923.37,14.15,False
10,2017-08-01,4193,554699.70,481604.52,15.18,False
11,2017-09-01,4150,607399.67,554699.70,9.50,False
12,2017-10-01,4478,648247.65,607399.67,6.73,False
13,2017-11-01,7289,987765.37,648247.65,52.37,False


In [16]:
# dejar el monitoreo listo para automatizar
# Seleccionar los meses con alertas
alertas = monitoreo[monitoreo["alerta"] == True].copy()

if alertas.empty:
    print("No se detectaron caídas superiores al 15 %.")
else:
    print(f"Se detectaron {len(alertas)} alerta(s):\n")

    for _, fila in alertas.iterrows():
        mes = fila["mes"].strftime("%m/%Y")
        variacion = fila["variacion_pct"]

        print(
            f"ALERTA | {mes}: "
            f"la facturación cayó un {abs(variacion):.2f}% "
            "respecto del mes anterior."
        )

Se detectaron 1 alerta(s):

ALERTA | 12/2017: la facturación cayó un 26.50% respecto del mes anterior.


In [17]:
def detectar_alertas(monitoreo, umbral_caida=-15):
    """
    Detecta caídas mensuales de facturación que superan
    el umbral definido, expresado como porcentaje.
    """
    alertas = monitoreo[
        monitoreo["variacion_pct"].notna()
        & (monitoreo["variacion_pct"] < umbral_caida)
    ].copy()

    return alertas

In [18]:
alertas = detectar_alertas(monitoreo)

if alertas.empty:
    print("No se detectaron caídas superiores al 15 %.")
else:
    print(f"Se detectaron {len(alertas)} alerta(s):\n")

    for _, fila in alertas.iterrows():
        mes = fila["mes"].strftime("%m/%Y")
        variacion = fila["variacion_pct"]

        print(
            f"ALERTA | {mes}: "
            f"la facturación cayó un {abs(variacion):.2f}% "
            "respecto del mes anterior."
        )

Se detectaron 1 alerta(s):

ALERTA | 12/2017: la facturación cayó un 26.50% respecto del mes anterior.


In [19]:
# separar la consulta a PostgreSQL de la lógica de monitoreo
def obtener_ventas_mensuales(engine):
    """
    Consulta PostgreSQL y devuelve los indicadores
    mensuales de ventas de Olist.
    """
    query = """
        SELECT
            DATE_TRUNC(
                'month', o.order_purchase_timestamp
            )::date AS mes,
            COUNT(DISTINCT o.order_id) AS pedidos_entregados,
            SUM(oi.price) AS facturacion_productos,
            SUM(oi.price) /
                NULLIF(COUNT(DISTINCT o.order_id), 0)
                AS ticket_promedio
        FROM orders AS o
        INNER JOIN order_items AS oi
            ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY
            DATE_TRUNC(
                'month', o.order_purchase_timestamp
            )::date
        ORDER BY mes;
    """

    return pd.read_sql(query, engine)

In [20]:
ventas_mensuales = obtener_ventas_mensuales(engine)

display(ventas_mensuales.head())
print(f"\nMeses obtenidos: {len(ventas_mensuales)}")

,mes,pedidos_entregados,facturacion_productos,ticket_promedio
0,2016-09-01,1,134.97,134.970000
1,2016-10-01,265,40325.11,152.170226
2,2016-12-01,1,10.90,10.900000
3,2017-01-01,750,111798.36,149.064480
4,2017-02-01,1653,234223.40,141.695947



Meses obtenidos: 23


In [21]:
# Crear la función principal de monitoreo
def ejecutar_monitoreo(engine, umbral_caida=-15):
    """
    Obtiene las ventas mensuales, calcula las variaciones
    y devuelve las alertas detectadas.
    """

    # 1. Extraer los datos desde PostgreSQL
    ventas = obtener_ventas_mensuales(engine)

    # 2. Preparar y ordenar los datos
    ventas["mes"] = pd.to_datetime(ventas["mes"])
    ventas = ventas.sort_values("mes").copy()

    # 3. Comparar con el mes calendario anterior
    facturacion_por_mes = ventas.set_index("mes")[
        "facturacion_productos"
    ]

    mes_anterior = ventas["mes"] - pd.DateOffset(months=1)

    ventas["facturacion_mes_anterior"] = mes_anterior.map(
        facturacion_por_mes
    )

    # 4. Calcular la variación porcentual
    ventas["variacion_pct"] = (
        ventas["facturacion_productos"]
        / ventas["facturacion_mes_anterior"]
        - 1
    ) * 100

    # 5. Detectar las caídas que superan el umbral
    alertas = detectar_alertas(
        ventas,
        umbral_caida=umbral_caida
    )

    return ventas, alertas

In [22]:
monitoreo, alertas = ejecutar_monitoreo(engine)

print(f"Meses analizados: {len(monitoreo)}")
print(f"Alertas detectadas: {len(alertas)}")

display(
    monitoreo[
        [
            "mes",
            "pedidos_entregados",
            "facturacion_productos",
            "variacion_pct",
        ]
    ].round(2)
)

for _, fila in alertas.iterrows():
    print(
        f"ALERTA | {fila['mes']:%m/%Y}: "
        f"caída del {abs(fila['variacion_pct']):.2f}% "
        "respecto del mes anterior."
    )

Meses analizados: 23
Alertas detectadas: 1


C:\Users\Josefina Urquiza\AppData\Local\Temp\ipykernel_14472\150290600.py:14: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,mes,pedidos_entregados,facturacion_productos,variacion_pct
0,2016-09-01,1,134.97,NaN
1,2016-10-01,265,40325.11,29777.09
2,2016-12-01,1,10.90,NaN
3,2017-01-01,750,111798.36,1025573.03
4,2017-02-01,1653,234223.40,109.51
5,2017-03-01,2546,359198.85,53.36
6,2017-04-01,2303,340669.68,-5.16
7,2017-05-01,3546,489338.25,43.64
8,2017-06-01,3135,421923.37,-13.78
9,2017-07-01,3872,481604.52,14.15


ALERTA | 12/2017: caída del 26.50% respecto del mes anterior.
